In [11]:
import sys

print(sys.executable)

c:\Users\Administrator\Desktop\siem-ai-project\ml\venv\Scripts\python.exe


In [12]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPUs Available:", len(tf.config.list_physical_devices('GPU')))

TensorFlow Version: 2.21.0
GPUs Available: 0


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense
)

In [18]:
df = pd.read_csv("../dataset/clean_dataset_sample.csv")

print(df.shape)

df.head()

(300000, 81)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,-1.341583,-0.425284,1.437675,-0.461085,-0.009935,-0.008393,-0.042407,-0.007051,-0.215421,0.422018,...,0.002268,-0.128428,-0.101914,-0.150163,-0.105006,-0.380578,-0.11786,-0.386236,-0.366398,0
1,0.676922,-0.423729,-0.694018,-0.313729,-0.008441,-0.009510,-0.050861,-0.007183,-0.273999,-0.271856,...,0.002278,-0.128428,-0.101914,-0.150163,-0.105006,-0.380578,-0.11786,-0.386236,-0.366398,0
2,-1.194689,-0.423729,-0.694018,-0.460889,-0.009935,-0.010627,-0.049758,-0.007183,-0.266359,-0.181351,...,0.002268,-0.128428,-0.101914,-0.150163,-0.105006,-0.380578,-0.11786,-0.386236,-0.366398,0
3,-0.063102,-0.388536,-0.694018,-0.461985,-0.009935,-0.010627,-0.050493,-0.007183,-0.271452,-0.241688,...,0.002272,-0.128428,-0.101914,-0.150163,-0.105006,-0.380578,-0.11786,-0.386236,-0.366398,0
4,0.843306,-0.402821,-0.694018,-0.456820,-0.002467,-0.005042,-0.016309,-0.005445,-0.018039,-0.271856,...,0.002268,-0.128428,-0.101914,-0.150163,-0.105006,-0.380578,-0.11786,-0.386236,-0.366398,0


In [19]:
X = df.drop("Label", axis=1)

y = df["Label"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(240000, 80)
(60000, 80)


In [21]:
X_train_normal = X_train[y_train == 0]

print(X_train_normal.shape)

(177482, 80)


In [22]:
X_train_lstm = np.expand_dims(
    X_train_normal.values,
    axis=1
)

X_test_lstm = np.expand_dims(
    X_test.values,
    axis=1
)

print(X_train_lstm.shape)

print(X_test_lstm.shape)

(177482, 1, 80)
(60000, 1, 80)


In [23]:
print(X_train_lstm.shape)
print(X_test_lstm.shape)

(177482, 1, 80)
(60000, 1, 80)


In [24]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

# Check shape
print("X_train_lstm shape:", X_train_lstm.shape)

timesteps = X_train_lstm.shape[1]
features = X_train_lstm.shape[2]

# Input layer
inputs = Input(shape=(timesteps, features))

# Encoder
x = LSTM(64, activation="relu", return_sequences=False)(inputs)

# Decoder
x = RepeatVector(timesteps)(x)

x = LSTM(64, activation="relu", return_sequences=True)(x)

outputs = TimeDistributed(Dense(features))(x)

# Autoencoder model
autoencoder = Model(inputs, outputs)

# Compile
autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

# Summary
autoencoder.summary()

X_train_lstm shape: (177482, 1, 80)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 80)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        37,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 80)          │         5,200 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,344 (294.31 KB)

 Trainable params: 75,344 (294.31 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train_lstm,
    X_train_lstm,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    shuffle=True,
    verbose=1
)

Epoch 1/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 75s 15ms/step - loss: 0.1508 - val_loss: 0.0122
Epoch 2/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 88s 20ms/step - loss: 0.0851 - val_loss: 0.0140
Epoch 3/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 95s 21ms/step - loss: 0.0640 - val_loss: 0.0079
Epoch 4/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 80s 18ms/step - loss: 0.0493 - val_loss: 0.0051
Epoch 5/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 69s 15ms/step - loss: 0.0437 - val_loss: 0.0064
Epoch 6/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 34s 8ms/step - loss: 0.0359 - val_loss: 0.0065
Epoch 7/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - loss: 0.0793 - val_loss: 0.0064
Epoch 8/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - loss: 0.0475 - val_loss: 0.0064
Epoch 9/50
4438/4438 ━━━━━━━━━━━━━━━━━━━━ 54s 12ms/step - loss: 0.0436 - val_loss: 0.0063


In [26]:
restore_best_weights=True

In [32]:
import os

os.makedirs("../models", exist_ok=True)

autoencoder.save("../models/lstm_autoencoder.keras")

print("Model Saved!")

Model Saved!
